# 00 · Quickstart: pipelines, configs, tracking
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/actuarialsociety/deep-actuarial-science/blob/main/notebooks/00_quickstart.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/actuarialsociety/deep-actuarial-science/blob/main/notebooks/00_quickstart.ipynb) [![Open In SageMaker Studio Lab](https://studiolab.sagemaker.aws/studiolab.svg)](https://studiolab.sagemaker.aws/import/github/actuarialsociety/deep-actuarial-science/blob/main/notebooks/00_quickstart.ipynb)

Run a pipeline, change a config, see it in MLflow. Every lesson works this way.

_Runs anywhere: **Colab** opens on a GPU runtime (T4) and the first cell stops on a CPU runtime; on **Kaggle** turn on *GPU T4* and *Internet* in the notebook settings; on **SageMaker Studio Lab** start a GPU runtime. In Azure ML, SageMaker Studio or any other Jupyter, clone the repo (or upload this notebook) and run all: the first cell installs what it needs._

In [ ]:
# Setup: runs anywhere Jupyter runs. Opened on its own (Colab, Kaggle, SageMaker, Azure ML,
# any hosted Jupyter), this cell clones and installs the repo; in a clone or the container it
# uses that checkout.
import os, sys, importlib.util
# Colab must run on a GPU runtime: stop here otherwise (set DAS_ALLOW_CPU=1 to override).
if "google.colab" in sys.modules and os.environ.get("DAS_ALLOW_CPU") != "1":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "This notebook needs a GPU runtime. In Colab: Runtime > Change runtime type > "
            "T4 GPU > Save, then Runtime > Run all. (To run on CPU anyway: "
            "import os; os.environ['DAS_ALLOW_CPU'] = '1')"
        )
    print("GPU:", torch.cuda.get_device_name(0))

REPO_DIR = next((d for d in (os.getcwd(), os.path.dirname(os.getcwd()))
                 if os.path.isdir(os.path.join(d, "src", "deep_actuarial_science"))), None)
if REPO_DIR is None:
    if sys.version_info < (3, 10):
        raise RuntimeError("Needs Python 3.10+: choose a newer kernel or image")
    # a writable, persistent home: Colab, Kaggle, SageMaker notebook instances, else ~
    base = next(d for d in ("/content", "/kaggle/working", os.path.expanduser("~/SageMaker"),
                            os.path.expanduser("~")) if os.path.isdir(d) and os.access(d, os.W_OK))
    REPO_DIR = os.path.join(base, "deep-actuarial-science")
    if not os.path.exists(REPO_DIR):  # Kaggle: turn on Internet in the notebook settings
        !git clone -q https://github.com/actuarialsociety/deep-actuarial-science {REPO_DIR}
    if importlib.util.find_spec("deep_actuarial_science") is None:
        !pip install -q -e {REPO_DIR}
os.chdir(REPO_DIR)
sys.path.insert(0, "src")
os.environ.setdefault("DO_NOT_TRACK", "1")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

from deep_actuarial_science.notebook import run, load, show, leaderboard, parameters, mlflow_ui, device_summary
device_summary()
FAST = os.environ.get("DAS_ENV") == "smoke"   # CI runs every notebook in smoke mode


## 1. Run a pipeline
The GLM and the homogeneous model are closed-form: this takes about a minute on the full 678k-policy dataset.

In [ ]:
run("frequency", models=["homogeneous", "glm"])
leaderboard("frequency", ["homogeneous", "glm"])

## 2. Reconcile to the published benchmark
Each model has a reconciliation table: **PASS** means this run reproduces the published result within tolerance.

In [ ]:
load("glm.freq_reconciliation")

## 3. Change a config, re-run
Parameters live in `conf/base/parameters_*.yml`. Override any key at run time, e.g. a different GLM formula:

In [ ]:
formula = "ClaimNb ~ DrivAgeGLM + BonusMalusGLM + VehBrand + VehGas + DensityGLM + AreaGLM"
run("frequency", models=["glm"], params={} if FAST else {"frequency.models.glm.formula": formula})
leaderboard("frequency", ["glm"])

## 4. Scale experiments
`frequency.data.scale` subsamples the data: check how a model's out-of-sample deviance and stability change with data volume.

In [ ]:
rows = []
for s in ([0.02] if FAST else [0.05, 0.25, 1.0]):
    run("frequency", models=["glm"], params={"frequency.data.scale": s})
    rows.append({"scale": s, **load("glm.freq_metrics")})
import pandas as pd; pd.DataFrame(rows)[["scale", "deviance_learn", "deviance_test", "gini_test"]]

## 5. Tracking
Every run above is in MLflow (params, metrics, figures, model pickle, benchmark table). Query it from Python below, or open the UI with `mlflow_ui()` in the last cell (works in Colab, the container and locally).

In [ ]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.search_runs(experiment_names=["deep-actuarial-science"]).filter(regex="start_time|metrics.glm.deviance").head()

## Explore the runs in MLflow
Parameters, metrics, figures and the benchmark table of every run above. MLflow runs on the notebook's machine: in Colab it opens below and in a new tab through Colab's proxy; in SageMaker and Azure ML through Jupyter's proxy; on Kaggle, which exposes no ports, query the runs with `mlflow.search_runs` instead.

In [ ]:
mlflow_ui()